# TA-004E — EfficientNet-B0 Reentrenamiento v2

**Objetivo:** Reentrenar EfficientNet-B0 con augmentation v2 e hiperparámetros óptimos encontrados por Optuna (TA-005E).

**Baseline anterior (TA-004B):** AUC test = 0.8862  
**Cambios:** Augmentation v2 (rotación ±180°, zoom, flip, colorjitter) + params Optuna  
**Meta:** superar AUC 0.8862

## 1. Imports

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Configuración

In [ ]:
BASE          = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
MANIFESTS     = BASE / 'dataset_split' / 'manifests'
TRAIN_DIR     = BASE / 'dataset_split' / 'train'
VAL_DIR       = BASE / 'dataset_split' / 'val'
TEST_DIR      = BASE / 'dataset_split' / 'test'
NOTEBOOKS_DIR = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
CKPT_DIR      = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')
CKPT_DIR.mkdir(exist_ok=True)

CLASSES     = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)
SEED        = 42
IMG_SIZE    = 224
BASELINE_AUC = 0.8862

with open(NOTEBOOKS_DIR / 'best_params_efficientnet.json') as f:
    best_params = json.load(f)

BATCH_SIZE   = 16
LR_FEATURE   = best_params['params']['lr_feature']
LR_FINETUNE  = best_params['params']['lr_finetune']
WEIGHT_DECAY = best_params['params']['weight_decay']
DROPOUT      = best_params['params']['dropout']
FE_EPOCHS    = 5
FT_EPOCHS    = 25
PATIENCE     = 10
STEP_SIZE    = max(1, FT_EPOCHS // 3)

WANDB_PROJECT = 'vetxray-cnn'
WANDB_ENTITY  = 'dbaylont1-antenor-orrego-private-university'

torch.manual_seed(SEED)
np.random.seed(SEED)

print('=== Config TA-004E ===')
print(f'batch_size  : {BATCH_SIZE}')
print(f'lr_feature  : {LR_FEATURE:.2e}')
print(f'lr_finetune : {LR_FINETUNE:.2e}')
print(f'weight_decay: {WEIGHT_DECAY:.2e}')
print(f'dropout     : {DROPOUT:.3f}')
print(f'scheduler   : StepLR (step={STEP_SIZE}, gamma=0.5)')
print(f'FE epochs   : {FE_EPOCHS} | FT epochs: {FT_EPOCHS} | patience: {PATIENCE}')
print(f'Baseline AUC: {BASELINE_AUC}')

## 3. W&B Init

In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name='TA-004E-EfficientNet-v2',
    config={
        'task':          'TA-004E',
        'model':         'efficientnet_b0',
        'batch_size':    BATCH_SIZE,
        'lr_feature':    LR_FEATURE,
        'lr_finetune':   LR_FINETUNE,
        'weight_decay':  WEIGHT_DECAY,
        'dropout':       DROPOUT,
        'scheduler':     'StepLR',
        'fe_epochs':     FE_EPOCHS,
        'ft_epochs':     FT_EPOCHS,
        'patience':      PATIENCE,
        'augmentation':  'v2',
        'baseline_auc':  BASELINE_AUC,
        'seed':          SEED
    },
    tags=['efficientnet', 'aug_v2', 'optuna_params', 'TA-004E', 'VetXRay']
)
print('W&B run iniciado:', wandb.run.url)

## 4. Transforms v2

In [ ]:
TRANSFORM_TRAIN = transforms.Compose([
    transforms.RandomRotation(degrees=180),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

TRANSFORM_VAL = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Augmentation v2: Rotation±180 | HFlip | VFlip | RandomResizedCrop(0.7-1.0) | ColorJitter')

## 5. Dataset y DataLoaders

In [ ]:
class VetXRayDataset(Dataset):
    def __init__(self, df, transform, base_dir):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.base_dir  = base_dir
        self.labels    = self._build_labels()

    def _build_labels(self):
        labels = np.zeros((len(self.df), NUM_CLASSES), dtype=np.float32)
        for i, row in self.df.iterrows():
            for j, cls in enumerate(CLASSES):
                if isinstance(row['TAG'], str) and cls in row['TAG']:
                    labels[i, j] = 1.0
        return labels

    def _load_dicom(self, path):
        dcm = pydicom.dcmread(str(path))
        arr = dcm.pixel_array.astype(np.float32)
        if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
            arr = arr.max() - arr
        p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
        arr = np.clip(arr, p2, p98)
        arr = (arr - p2) / (p98 - p2 + 1e-8)
        img = Image.fromarray((arr * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def _load_png(self, path):
        img = Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if row.get('is_synthetic', False) and pd.notna(row.get('synthetic_path')):
            img = self._load_png(row['synthetic_path'])
        else:
            img = self._load_dicom(self.base_dir / row['FileName'])
        return self.transform(img), torch.tensor(self.labels[idx], dtype=torch.float32)

print('VetXRayDataset OK')

In [ ]:
df_train = pd.read_csv(MANIFESTS / 'train_augmented.csv')
df_val   = pd.read_csv(MANIFESTS / 'val.csv')
df_test  = pd.read_csv(MANIFESTS / 'test.csv')

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

pos_counts    = np.array([df_train['TAG'].str.contains(c, na=False).sum() for c in CLASSES], dtype=np.float32)
neg_counts    = len(df_train) - pos_counts
CLASS_WEIGHTS = torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32).to(DEVICE)

print('\nClass weights:')
for c, w in zip(CLASSES, CLASS_WEIGHTS.cpu()):
    print(f'  {c:<22}: {w:.3f}')

train_ds = VetXRayDataset(df_train, TRANSFORM_TRAIN, TRAIN_DIR)
val_ds   = VetXRayDataset(df_val,   TRANSFORM_VAL,   VAL_DIR)
test_ds  = VetXRayDataset(df_test,  TRANSFORM_VAL,   TEST_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'\nDataLoaders OK — batch_size={BATCH_SIZE}')

## 6. Modelo

In [ ]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
for p in model.parameters(): p.requires_grad = False
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_features, NUM_CLASSES))
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B0 | Dropout={DROPOUT:.3f}')
print(f'Params entrenables: {trainable:,} / {total:,}')

## 7. Entrenamiento

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=CLASS_WEIGHTS)
scaler    = torch.amp.GradScaler('cuda')
history   = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_auc  = 0.0
best_epoch = 0

def run_epoch(loader, train=True, optimizer=None):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    loss = criterion(model(imgs), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                with torch.amp.autocast('cuda'):
                    out  = model(imgs)
                    loss = criterion(out, labels)
                all_probs.append(torch.sigmoid(out).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
            total_loss += loss.item() * imgs.size(0)
    avg_loss = total_loss / len(loader.dataset)
    if not train:
        P = np.vstack(all_probs); L = np.vstack(all_labels)
        try: auc = roc_auc_score(L, P, average='macro')
        except: auc = 0.0
        f1 = f1_score(L, (P > 0.5).astype(int), average='macro', zero_division=0)
        return avg_loss, auc, f1
    return avg_loss

print('Training helpers OK')

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_FEATURE, weight_decay=WEIGHT_DECAY
)

print(f'=== FASE 1: Feature Extraction ({FE_EPOCHS} épocas) ===')
for ep in range(1, FE_EPOCHS + 1):
    t0         = time.time()
    train_loss = run_epoch(train_loader, train=True, optimizer=optimizer)
    val_loss, val_auc, val_f1 = run_epoch(val_loader, train=False)
    elapsed    = (time.time() - t0) / 60
    lr_now     = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)

    wandb.log({'epoch': ep, 'phase': 'FE',
               'train_loss': train_loss, 'val_loss': val_loss,
               'val_auc': val_auc, 'val_f1': val_f1, 'lr': lr_now})

    marker = ''
    if val_auc > best_auc:
        best_auc   = val_auc
        best_epoch = ep
        torch.save(model.state_dict(), CKPT_DIR / 'best_efficientnet_v2.pth')
        marker = ' >>> Mejor AUC guardado'

    print(f'[FE] Ep {ep:2d}/{FE_EPOCHS} | '
          f'Train {train_loss:.4f} | Val {val_loss:.4f} | '
          f'AUC {val_auc:.4f} | F1 {val_f1:.4f} | '
          f'LR {lr_now:.2e} | {elapsed:.1f}min{marker}', flush=True)

In [ ]:
for p in model.parameters(): p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=0.5)

patience_counter = 0
print(f'Params entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'=== FASE 2: Fine-Tuning ({FT_EPOCHS} épocas | StepLR step={STEP_SIZE}) ===')

for ep in range(FE_EPOCHS + 1, FE_EPOCHS + FT_EPOCHS + 1):
    t0         = time.time()
    train_loss = run_epoch(train_loader, train=True, optimizer=optimizer)
    val_loss, val_auc, val_f1 = run_epoch(val_loader, train=False)
    scheduler.step()
    elapsed = (time.time() - t0) / 60
    lr_now  = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)

    wandb.log({'epoch': ep, 'phase': 'FT',
               'train_loss': train_loss, 'val_loss': val_loss,
               'val_auc': val_auc, 'val_f1': val_f1, 'lr': lr_now})

    marker = ''
    if val_auc > best_auc:
        best_auc     = val_auc
        best_epoch   = ep
        patience_counter = 0
        torch.save(model.state_dict(), CKPT_DIR / 'best_efficientnet_v2.pth')
        marker = ' >>> Mejor AUC guardado'
    else:
        patience_counter += 1

    print(f'[FT] Ep {ep:2d}/{FE_EPOCHS+FT_EPOCHS} | '
          f'Train {train_loss:.4f} | Val {val_loss:.4f} | '
          f'AUC {val_auc:.4f} | F1 {val_f1:.4f} | '
          f'LR {lr_now:.2e} | {elapsed:.1f}min{marker}', flush=True)

    if patience_counter >= PATIENCE:
        print(f'Early stopping en época {ep} (patience={PATIENCE})')
        break

print(f'\nEntrenamiento completo. Mejor AUC val: {best_auc:.4f} (época {best_epoch})')

## 8. Evaluación en Test

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best_efficientnet_v2.pth', map_location=DEVICE))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        with torch.amp.autocast('cuda'):
            probs = torch.sigmoid(model(imgs.to(DEVICE)))
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

probs_test  = np.vstack(all_probs)
labels_test = np.vstack(all_labels)
preds_test  = (probs_test > 0.5).astype(int)

test_auc = roc_auc_score(labels_test, probs_test, average='macro')
test_f1  = f1_score(labels_test, preds_test, average='macro', zero_division=0)

print(f'=== Resultados Test ===')
print(f'AUC macro : {test_auc:.4f}  (baseline: {BASELINE_AUC}  |  delta: {test_auc-BASELINE_AUC:+.4f})')
print(f'F1  macro : {test_f1:.4f}')
print()
print(f'{"Clase":<22} {"AUC":>8} {"F1":>8}')
print('-' * 40)
for i, cls in enumerate(CLASSES):
    try: auc_c = roc_auc_score(labels_test[:, i], probs_test[:, i])
    except: auc_c = 0.0
    f1_c = f1_score(labels_test[:, i], preds_test[:, i], zero_division=0)
    print(f'{cls:<22} {auc_c:>8.4f} {f1_c:>8.4f}')

wandb.summary.update({
    'test_auc':     round(test_auc, 4),
    'test_f1':      round(test_f1, 4),
    'best_val_auc': round(best_auc, 4),
    'best_epoch':   best_epoch,
    'delta_auc':    round(test_auc - BASELINE_AUC, 4)
})

## 9. W&B Charts

In [ ]:
epochs_range = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TA-004E EfficientNet-B0 v2 — Curvas de entrenamiento', fontsize=12)

axes[0].plot(epochs_range, history['train_loss'], label='Train loss')
axes[0].plot(epochs_range, history['val_loss'],   label='Val loss')
axes[0].axvline(FE_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='FE→FT')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss por época'); axes[0].legend()

axes[1].plot(epochs_range, history['val_auc'], label='Val AUC', color='orange')
axes[1].plot(epochs_range, history['val_f1'],  label='Val F1',  color='green')
axes[1].axhline(BASELINE_AUC, color='red', linestyle='--', alpha=0.7, label=f'Baseline {BASELINE_AUC}')
axes[1].axvline(FE_EPOCHS, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Score')
axes[1].set_title('AUC y F1 por época'); axes[1].legend()

plt.tight_layout()
curves_path = NOTEBOOKS_DIR / 'ta004e_curves.png'
plt.savefig(str(curves_path), dpi=100, bbox_inches='tight')
wandb.log({'training_curves': wandb.Image(str(curves_path))})
plt.show()

auc_per_class = []
for i, cls in enumerate(CLASSES):
    try: auc_per_class.append(roc_auc_score(labels_test[:, i], probs_test[:, i]))
    except: auc_per_class.append(0.0)

fig2, ax2 = plt.subplots(figsize=(10, 5))
bars = ax2.barh(CLASSES, auc_per_class, color='steelblue')
ax2.axvline(BASELINE_AUC, color='red', linestyle='--', label=f'Baseline {BASELINE_AUC}')
ax2.axvline(test_auc, color='orange', linestyle='--', label=f'AUC macro {test_auc:.4f}')
for bar, val in zip(bars, auc_per_class):
    ax2.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')
ax2.set_xlim(0, 1.1); ax2.set_title('AUC por clase — Test'); ax2.legend()
plt.tight_layout()
auc_path = NOTEBOOKS_DIR / 'ta004e_auc_clases.png'
plt.savefig(str(auc_path), dpi=100, bbox_inches='tight')
wandb.log({'auc_por_clase': wandb.Image(str(auc_path))})
plt.show()

wandb.log({
    'metricas_test': wandb.Table(
        columns=['clase', 'auc', 'f1'],
        data=[
            [cls,
             round(roc_auc_score(labels_test[:, i], probs_test[:, i]), 4) if labels_test[:, i].sum() > 0 else 0.0,
             round(f1_score(labels_test[:, i], preds_test[:, i], zero_division=0), 4)]
            for i, cls in enumerate(CLASSES)
        ] + [['MACRO', round(test_auc, 4), round(test_f1, 4)]]
    )
})

wandb.finish()
print('W&B cerrado. TA-004E completado.')

## 10. Guardado de resultados

In [ ]:
results = {
    'model':        'efficientnet_b0',
    'task':         'TA-004E',
    'augmentation': 'v2',
    'test_auc':     round(test_auc, 4),
    'test_f1':      round(test_f1, 4),
    'best_val_auc': round(best_auc, 4),
    'best_epoch':   best_epoch,
    'baseline_auc': BASELINE_AUC,
    'delta_auc':    round(test_auc - BASELINE_AUC, 4),
    'params': {
        'batch_size':   BATCH_SIZE,
        'lr_feature':   LR_FEATURE,
        'lr_finetune':  LR_FINETUNE,
        'weight_decay': WEIGHT_DECAY,
        'dropout':      DROPOUT,
        'scheduler':    'StepLR',
        'fe_epochs':    FE_EPOCHS,
        'ft_epochs':    FT_EPOCHS
    }
}
with open(NOTEBOOKS_DIR / 'efficientnet_v2_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print(f'Resultados guardados.')
print(f'AUC test : {test_auc:.4f}')
print(f'Baseline : {BASELINE_AUC}')
print(f'Delta    : {test_auc - BASELINE_AUC:+.4f}')
print(f'Checkpoint: checkpoints/best_efficientnet_v2.pth')

## 11. Conclusiones

**TA-004E completado.** EfficientNet-B0 reentrenado con:
- Augmentation v2: rotación ±180°, flip H/V, RandomResizedCrop, ColorJitter
- Hiperparámetros optimizados por Optuna (TA-005E)
- Scheduler StepLR en fase fine-tuning

**Siguiente paso:** TA-004F — DenseNet-121 con misma configuración.